The LLM architecture consists of several core components that are repeated throughout the model. Before implementing the complete LLM, we will first understand each of these components individually and then combine them to build the GPT architecture.

A GPT transformer block is composed of several key building blocks, including Layer Normalization, Multi-Head Attention (covered in the previous chapter), a Feed-Forward Neural Network (FFN), GELU activation, and Shortcut (Residual) Connections. Understanding how each of these components works is essential before implementing the complete GPT model.

In this section, we will implement layer normalization to improve the stability and efficiency of neural network training.
The main idea behind layer normalization is to adjust the activations (outputs) of a neural network layer to have a mean of 0 and a variance of 1, also known as unit variance.
This adjustment speeds up the convergence to effective weights and ensures consistent, reliable training.

Without normalization, the output of a layer can drift during training.

For example:

Layer 1 outputs values around 0.5
After several updates, they become around 50
Later they might become 0.0001

Large activations can saturate nonlinearities (or produce huge gradients), while tiny activations can lead to vanishing gradients. Both make optimization difficult.

Layer Normalization rescales every input to approximately

μ=0,σ^2=1

so each layer always receives inputs of a similar scale.

# Layer Normalization

To normalize a vector of numbers, we first compute its **mean** and **variance**. We then subtract the mean from each element and divide by the **standard deviation**.

For example, consider the vector

$$
x = [1, 5, 90, -20]
$$

### Step 1: Compute the mean

$$
\mu = \frac{1 + 5 + 90 - 20}{4} = 19
$$

### Step 2: Center the values

$$
x - \mu = [1-19,\;5-19,\;90-19,\;-20-19]
$$

$$
= [-18,\;-14,\;71,\;-39]
$$

### Step 3: Compute the variance

$$
\sigma^2 = \frac{(-18)^2 + (-14)^2 + 71^2 + (-39)^2}{4}
= 1765.5
$$

### Step 4: Compute the standard deviation

$$
\sigma = \sqrt{1765.5} \approx 42.02
$$

### Step 5: Normalize the vector

$$
\hat{x} = \frac{x - \mu}{\sigma}
$$

$$
=
\left[
\frac{-18}{42.02},
\frac{-14}{42.02},
\frac{71}{42.02},
\frac{-39}{42.02}
\right]
$$

$$
\approx [-0.43,\;-0.33,\;1.69,\;-0.93]
$$

The normalized vector has:

- **Mean = 0**
- **Variance = 1**

In [2]:
pip install torch


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().init()
        self.eps = 1e-5 # small eps to avoid divisible by zero variance
        self.scale = nn.parameter(torch.ones(emb_dim))
        self.shift = nn.parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False) # does not apply Bessel's correction
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return  self.scale * norm_x + self.shift
        

/usr/local/python/3.12.1/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
